# 📊 EDA – Multimodal Dataset (CIFAR-100 Superclasses + CLIP)

**CO5085 – Assignment 1**

Mục đích: Khám phá tập dữ liệu đa phương thức dùng CIFAR-100 superclasses làm proxy với text prompts tự động.

In [ ]:
import sys
sys.path.append('../src')
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch, clip
from torchvision import datasets
import torchvision.transforms as T
from collections import Counter
import os
os.makedirs('../results', exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Libraries loaded ✅ | Device: {DEVICE}')

## 1. Dataset Overview

In [ ]:
# CIFAR-100 superclasses (20 groups used as multimodal class labels)
SUPERCLASSES = [
    'aquatic mammals', 'fish', 'flowers', 'food containers', 'fruit and vegetables',
    'household electrical devices', 'household furniture', 'insects', 'large carnivores',
    'large man-made outdoor things', 'large natural outdoor scenes', 'large omnivores and herbivores',
    'medium-sized mammals', 'non-insect invertebrates', 'people', 'reptiles',
    'small mammals', 'trees', 'vehicles 1', 'vehicles 2'
]

coarse_labels = [
    4, 1, 14, 8, 0, 6, 7, 7, 18, 3, 3, 14, 9, 18, 7, 11, 3, 9, 7, 11,
    6, 11, 5, 10, 7, 6, 13, 15, 3, 15, 0, 11, 1, 10, 12, 14, 16, 9, 11, 5,
    5, 19, 8, 8, 15, 13, 14, 17, 18, 10, 16, 4, 17, 4, 2, 0, 17, 4, 18, 17,
    10, 3, 2, 12, 12, 16, 12, 1, 9, 19, 2, 10, 0, 1, 16, 12, 9, 13, 15, 13,
    16, 19, 2, 4, 6, 19, 5, 5, 8, 19, 18, 1, 2, 15, 6, 0, 17, 8, 14, 13
]

train_ds = datasets.CIFAR100(root='../data/image', train=True, download=True,
                              transform=T.ToTensor())
test_ds  = datasets.CIFAR100(root='../data/image', train=False, download=True,
                              transform=T.ToTensor())
print(f'Train: {len(train_ds)} | Test: {len(test_ds)} | Superclasses: {len(SUPERCLASSES)}')

## 2. Superclass Distribution

In [ ]:
train_coarse = [coarse_labels[y] for _, y in train_ds]
counts = Counter(train_coarse)

plt.figure(figsize=(14, 5))
plt.bar([SUPERCLASSES[i] for i in sorted(counts)], [counts[i] for i in sorted(counts)], color='mediumpurple')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.title('CIFAR-100 Superclass Distribution (Train)')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('../results/multimodal_superclass_dist.png', dpi=150)
plt.show()
print(f'Samples per superclass – Min: {min(counts.values())} | Max: {max(counts.values())}')

## 3. Sample Image-Text Pairs

In [ ]:
fig, axes = plt.subplots(4, 5, figsize=(15, 12))
shown = {}
for img_t, fine_label in train_ds:
    sc = coarse_labels[fine_label]
    if sc not in shown:
        shown[sc] = (img_t, SUPERCLASSES[sc])
    if len(shown) == 20:
        break

for ax, (sc_idx, (img_t, prompt)) in zip(axes.flat, sorted(shown.items())):
    ax.imshow(img_t.permute(1, 2, 0).clamp(0, 1))
    ax.set_title(f'\'a photo of a {prompt}\'', fontsize=7, wrap=True)
    ax.axis('off')

plt.suptitle('Image–Text Pairs (one sample per superclass)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/multimodal_image_text_pairs.png', dpi=150)
plt.show()

## 4. Text Prompt Statistics (CLIP Tokenizer)

In [ ]:
# Load CLIP tokenizer (no model weights needed for this)
model_clip, preprocess = clip.load('ViT-B/32', device='cpu')

prompts = [f'a photo of a {sc}' for sc in SUPERCLASSES]
token_lengths = [len(clip.tokenize(p)[0].nonzero()) for p in prompts]

print('Text prompt token lengths:')
for prompt, tlen in zip(prompts, token_lengths):
    print(f'  [{tlen:2d} tokens] {prompt}')
print(f'\nMean: {np.mean(token_lengths):.1f} | Max: {max(token_lengths)}')

## 5. CLIP Embedding Space Visualization (PCA)

In [ ]:
from sklearn.decomposition import PCA

# Encode all 20 text prompts
with torch.no_grad():
    tokens = clip.tokenize(prompts).to('cpu')
    text_features = model_clip.encode_text(tokens).float().numpy()

pca = PCA(n_components=2)
coords = pca.fit_transform(text_features)

plt.figure(figsize=(10, 7))
for i, (x, y) in enumerate(coords):
    plt.scatter(x, y, s=80, color='coral')
    plt.annotate(SUPERCLASSES[i], (x, y), fontsize=8, textcoords='offset points', xytext=(5, 3))

plt.title('CLIP Text Embeddings – PCA (20 Superclass Prompts)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.tight_layout()
plt.savefig('../results/multimodal_clip_text_pca.png', dpi=150)
plt.show()
print(f'Explained variance: PC1={pca.explained_variance_ratio_[0]:.2%}, PC2={pca.explained_variance_ratio_[1]:.2%}')